# 문제1

In [ ]:
import torch

model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)

In [ ]:
from collections import Counter

total_counter = Counter()

In [ ]:
from PIL import Image

img_path = './images/surv_imgs/CAM001.png'
img = Image.open(img_path)

In [ ]:
results = model(img)

In [ ]:
results

In [ ]:
preds = results.pandas().xyxy[0]['name'].tolist()

In [ ]:
preds

In [ ]:
total_counter.update(preds)

In [ ]:
total_counter

In [ ]:
import pandas as pd

df = pd.DataFrame(total_counter.items(), columns=["object_class", "count"])
df = df.sort_values(by="count", ascending=False)
df.to_csv("object_count.csv", index=False)

In [ ]:
df

In [ ]:
df.head(1)

In [ ]:
import matplotlib.pyplot as plt

top3 = df.head(3)

In [ ]:
top3

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(top3["object_class"], top3["count"])
plt.title("Top 3 Detected Objects")
plt.xlabel("Object Class")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("top3_objects.png")

### 정리

In [ ]:
import os
import torch
from collections import Counter
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt

# 모델 로드
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)

# 이미지 디렉토리 경로
img_dir = './images/surv_imgs/'
img_files = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

# 전체 클래스 카운터
total_counter = Counter()

# 1. 이미지 순회하며 객체 탐지
for img_file in img_files:
    img_path = os.path.join(img_dir, img_file)
    img = Image.open(img_path)
    results = model(img)
    preds = results.pandas().xyxy[0]['name'].tolist()
    total_counter.update(preds)

# 2. 결과 CSV 저장
df = pd.DataFrame(total_counter.items(), columns=["object_class", "count"])
df = df.sort_values(by="count", ascending=False)
df.to_csv("object_count.csv", index=False)

# 3. 상위 3개 클래스 시각화
top3 = df.head(3)
plt.figure(figsize=(8, 5))
plt.bar(top3["object_class"], top3["count"])
plt.title("Top 3 Detected Objects")
plt.xlabel("Object Class")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("top3_objects.png")
plt.close()

---

# 문제2

In [ ]:
!pip install easyocr

In [ ]:
import easyocr

reader = easyocr.Reader(['ko', 'en'])

In [ ]:
img = "./images/supplies_imgs/BOX001.png"

In [ ]:
results = []

In [ ]:
result = reader.readtext(img, detail=0)

In [ ]:
result

In [ ]:
lines = [line.strip() for line in result if line.strip()]

In [ ]:
lines

In [ ]:
import re

for line in lines:
    match = re.match(r"([가-힣a-zA-Z0-9]+)", line)
    if match:
        results.append({"filename": img, "item": line})

In [ ]:
results

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df.to_csv("supply_info.csv", index=False)

In [ ]:
df

### 정리

In [ ]:
import os
import cv2
import re
import pandas as pd
import easyocr

# EasyOCR Reader 초기화 (한글 + 영어)
reader = easyocr.Reader(['ko', 'en'])

# 이미지 디렉토리 설정
img_dir = "./images/supplies_imgs/"
img_files = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

# 결과 저장 리스트 생성
results = []

# 1. 이미지 OCR 및 정규표현식 파싱
for img_file in img_files:
    img_path = os.path.join(img_dir, img_file)

    # 1. OCR로 텍스트 추출
    result = reader.readtext(img_path, detail=0)  # detail=0은 텍스트만 반환
    lines = [line.strip() for line in result if line.strip()]

    for line in lines:
        # 2. 정규 표현식 활용
        match = re.match(r"([가-힣a-zA-Z0-9]+)", line)
        if match:
            results.append({"filename": img, "item": line})

# 3. 결과 저장
df = pd.DataFrame(results)
df.to_csv("supply_info.csv", index=False)

---

# 문제3

In [ ]:
import torch

model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)

In [ ]:
import cv2

video_path = './videos/drone_mission.mp4'
cap = cv2.VideoCapture(video_path)

In [ ]:
fps = cap.get(cv2.CAP_PROP_FPS)
frame_interval = int(fps)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

In [ ]:
results_list = []

frame_num = 0
read_frame_num = 0

In [ ]:
from collections import defaultdict

detection_count_by_frame = defaultdict(int)

In [ ]:
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    if frame_num % frame_interval == 0:
        # YOLO 추론
        results = model(frame)
        df = results.pandas().xyxy[0]

        for _, row in df.iterrows():
            results_list.append({
                "frame_number": frame_num,
                "class": row["name"],
                "x1": int(row["xmin"]),
                "y1": int(row["ymin"]),
                "x2": int(row["xmax"]),
                "y2": int(row["ymax"])
            })
            detection_count_by_frame[frame_num] += 1

        read_frame_num += 1
        print(f"{read_frame_num}번째 추출 프레임(frame {frame_num}) 분석 완료")

    frame_num += 1

cap.release()

In [ ]:
import pandas as pd

df_result = pd.DataFrame(results_list)
df_result.to_csv("drone_detection.csv", index=False)

In [ ]:
df_result

In [ ]:
import matplotlib.pyplot as plt

frame_series = pd.Series(detection_count_by_frame)
frame_series.index.name = 'frame_number'
frame_series.name = 'detected_objects'

plt.figure(figsize=(10, 5))
frame_series.sort_index().plot(marker='o')
plt.title("YOLO Detections per Frame (1s Interval)")
plt.xlabel("Frame Number")
plt.ylabel("Number of Detections")
plt.grid(True)
plt.tight_layout()
plt.savefig("detection_over_time.png")
plt.close()

### 정리

In [ ]:
import cv2
import torch
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

# YOLOv5 모델 로드
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)

# 영상 로드
video_path = './videos/drone_mission.mp4'
cap = cv2.VideoCapture(video_path)

# 1. 프레임 추출
fps = cap.get(cv2.CAP_PROP_FPS)
frame_interval = int(fps)  # 1초마다 추출
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

results_list = []
detection_count_by_frame = defaultdict(int)

frame_num = 0
read_frame_num = 0

# 2. 객체 탐지 수행
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    if frame_num % frame_interval == 0:
        # YOLO 추론
        results = model(frame)
        df = results.pandas().xyxy[0]  # 객체별 x1, y1, x2, y2, name 등

        for _, row in df.iterrows():
            results_list.append({
                "frame_number": frame_num,
                "class": row["name"],
                "x1": int(row["xmin"]),
                "y1": int(row["ymin"]),
                "x2": int(row["xmax"]),
                "y2": int(row["ymax"])
            })
            detection_count_by_frame[frame_num] += 1

        read_frame_num += 1
        print(f"{read_frame_num}번째 추출 프레임(frame {frame_num}) 분석 완료")

    frame_num += 1

cap.release()

# 3. 탐지 결과 CSV 저장
df_result = pd.DataFrame(results_list)
df_result.to_csv("drone_detection.csv", index=False)

# 4. 프레임별 탐지 수 시계열 그래프
frame_series = pd.Series(detection_count_by_frame)
frame_series.index.name = 'frame_number'
frame_series.name = 'detected_objects'

plt.figure(figsize=(10, 5))
frame_series.sort_index().plot(marker='o')
plt.title("YOLO Detections per Frame (1s Interval)")
plt.xlabel("Frame Number")
plt.ylabel("Number of Detections")
plt.grid(True)
plt.tight_layout()
plt.savefig("detection_over_time.png")
plt.close()